# 52. The task-drift probe on the model whose behaviour it is compared with (Qwen2.5-3B)

Notebook 27 fitted the task-drift probe on the hidden states of Qwen2.5-7B-Instruct and then compared its flags with judge verdicts on the outputs of the Section 8 target, Qwen2.5-3B-Instruct. Its description called the 7B "the same target as the adversary"; it was not. So the cross-tab in Table 27 of the paper pairs one model's internal state with another model's behaviour, and the paper now says so. This notebook repeats the probe on the 3B itself, with everything else unchanged, so that the question the cross-tab asks can be answered on one model: among the rewrites the 3B actually followed, how many does a probe on the 3B's own states flag?

**Unchanged from notebook 27.** The probe's training data (600 and 600 structural records for training, 150 and 150 for validation, drawn with the same seed in the same order), the evaluation sets (BIPIA, the same 600 second-source records, NotInject, 200 recipes, the same 129 benign and 400 attacked AgentDojo records, the 455 adversarial variants of Section 8), the task prompt and echo, the pooling of the hidden states over the repeated task, the logistic-regression probe with its three regularization strengths, the layer chosen by validation AUROC, and the judge verdicts on disk (`data/phase5v2/judge_7b.json`), which are the 3B target's.

**Changed.** The model read is Qwen2.5-3B-Instruct in 4-bit, as in Section 8. The three candidate layers sit at the same depths as notebook 27's 8, 14 and 20 of 28: layers 10, 18 and 26 of 36. Every guaranteed threshold uses the paper's final routine (`strict_threshold`, Appendix A.2), and the AgentDojo benigns also get the leave-one-out false-positive estimate of Table 16. For comparison, the 7B probe is refitted from notebook 27's cached features and scored under the same routine, so the two models' rows are computed by one piece of code.

**What the paper takes from it.** Section 9.4 and Table 27 are rewritten around the 3B rows; the 7B rows stay as the comparison; the cross-tab with Wilson intervals replaces the 7B cross-tab; the sentence "whether it sees that a model will obey it was not tested on one model" is replaced by whatever this notebook finds, favourable or not.

GPU. About 5,000 forward passes of the 3B; roughly 25 minutes on an A100, all cached. Rerunning skips every cached stage.

**Outputs.** `data/taskdrift_3b/feat_*.npy`, `reports/taskdrift3b_eval.csv`, `reports/taskdrift3b_adversary.csv`, `reports/taskdrift3b_crosstab.csv`, `reports/taskdrift3b_tasks.csv`, `reports/taskdrift3b_layers.csv`, `figures/taskdrift_3b.png`.

In [ ]:
# === SESSION BOOTSTRAP (run first): install from the local disk, then mount ===
!pip install -q transformers datasets accelerate sentencepiece bitsandbytes
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
for d_ in ('reports', 'figures', 'data/taskdrift_3b'): os.makedirs(d_, exist_ok=True)
import torch, gc
assert torch.cuda.is_available(), 'this notebook needs a GPU'
print('Session ready. GPU:', torch.cuda.get_device_name(0))

## Stage A. The same inputs as notebook 27, in the same order

In [ ]:
import importlib, json, numpy as np, pandas as pd
import taskdrift, phase5_redo, data_loaders, structural_train, strict_threshold, ci_tools
[importlib.reload(m) for m in (taskdrift, phase5_redo, data_loaders, structural_train, strict_threshold, ci_tools)]
from taskdrift import extract_features, fit_probe, probe_scores, TASKS
from phase5_redo import flat_variants, group_min_index
from data_loaders import load_bipia_categorized, load_notinject, BIPIA_HARMFUL, BIPIA_HIJACK, BIPIA_MALICIOUS
from structural_train import load_jsonl
from strict_threshold import strict_threshold as guar_threshold, loo_fpr, fnr
from ci_tools import boot_auroc, wilson, fmt
from sklearn.metrics import roc_auc_score
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
# notebook 27's random draws, in notebook 27's order, so the sets are the same records
rng = np.random.default_rng(0)
def probe_set(path, n_pos, n_neg):
    R = [r for r in load_jsonl(path) if r['form'] != 'standalone']; pos = [r for r in R if r['label'] == 1]; neg = [r for r in R if r['label'] == 0]
    pos = [pos[i] for i in rng.choice(len(pos), min(n_pos, len(pos)), replace=False)]; neg = [neg[i] for i in rng.choice(len(neg), min(n_neg, len(neg)), replace=False)]
    return [r['text'] for r in pos + neg], np.array([1] * len(pos) + [0] * len(neg))
TR_X, TR_Y = probe_set('data/structural/train.jsonl', 600, 600); VA_X, VA_Y = probe_set('data/structural/val.jsonl', 150, 150)
bip = load_bipia_categorized(per_category=30, categories=BIPIA_MALICIOUS); HOST = bip.label.values == 0; HIJ = bip.meta.isin(BIPIA_HIJACK).values; HARM = bip.meta.isin(BIPIA_HARMFUL).values
OPI = json.load(open('data/opi/opi_eval.json'))['records']; oi = rng.choice(len(OPI), 600, replace=False); OPI_X = [OPI[i]['text'] for i in oi]; OPI_Y = np.array([OPI[i]['label'] for i in oi])
noti = load_notinject(); HOW = json.load(open('data/closers/howto.json'))['texts'][:200]
AD = json.load(open('data/agentdojo/set.json')); AD_b = [r for r in AD if r['label'] == 0]; AD_a = [r for r in AD if r['label'] == 1 and r.get('inj_offset', 0) <= 1600]; AD_a = [AD_a[i] for i in rng.choice(len(AD_a), 400, replace=False)]
AD_X = [r['text'] for r in AD_b + AD_a]; AD_Y = np.array([0] * len(AD_b) + [1] * len(AD_a))
SUB = pd.DataFrame(json.load(open('data/phase5v2/subset.json'))); REW = json.load(open('data/phase5v2/rewrites.json')); VAR, OWNER, FIRST = flat_variants(SUB, REW)
SETS = {'train': TR_X, 'val': VA_X, 'bipia': bip.text.tolist(), 'opi': OPI_X, 'notinject': noti.text.tolist(), 'howto': HOW, 'agentdojo': AD_X, 'adv': VAR}
print({k: len(v) for k, v in SETS.items()})
# the sets must be the ones notebook 27 cached: same sizes and, for the 7B features, same record counts
OLD = {s: f'data/taskdrift/feat_summarize_{s}_L14.npy' for s in SETS}
assert all(os.path.exists(f) for f in OLD.values()), 'notebook 27 features missing: this notebook compares with them'
for s, f in OLD.items(): assert np.load(f, mmap_mode='r').shape[0] == len(SETS[s]), f'{s}: {len(SETS[s])} records now, notebook 27 cached {np.load(f, mmap_mode="r").shape[0]}'
JV = json.load(open('data/phase5v2/judge_7b.json'))
SRCS = ('protectai_v2', 'prompt_guard_2', 'prompt_guard_2_22m', 'full', 'full_aug')
SELS = {src: group_min_index(np.load(f'data/phase5v2/score_{src}.npy'), OWNER) for src in SRCS}
assert all(k.split('|')[0] in SRCS + ('orig',) for k in JV), 'unexpected judge keys'
print('judge verdicts on disk (the 3B target\'s outputs):', len(JV), '| sets match notebook 27')

## Stage B. Hidden-state features from the 3B (cached; the slow stage)

In [ ]:
from targets import load_lm
MODELS = {'3b': dict(name='Qwen/Qwen2.5-3B-Instruct', layers=[10, 18, 26], feat_dir='data/taskdrift_3b', label='Qwen2.5-3B (the Section 8 target)'),
          '7b': dict(name='Qwen/Qwen2.5-7B-Instruct', layers=[8, 14, 20], feat_dir='data/taskdrift', label='Qwen2.5-7B (notebook 27)')}
TASK = 'summarize'
def feat_path(m, s, L, task=TASK): return f"{MODELS[m]['feat_dir']}/feat_{task}_{s}_L{L}.npy"
m_ = '3b'; LAYERS = MODELS[m_]['layers']
todo = [s for s in SETS if not all(os.path.exists(feat_path(m_, s, L)) for L in LAYERS)]
todo_tasks = [(tk, 'bipia') for tk in ('qa', 'extract') if not all(os.path.exists(feat_path(m_, 'bipia', L, tk)) for L in LAYERS)]
if todo or todo_tasks:
    tm, tt = load_lm(MODELS[m_]['name'], four_bit=True); nL = tm.config.num_hidden_layers; print('layers available:', nL)
    assert nL == 36 and max(LAYERS) <= nL, nL   # hidden_states[0] is the embedding output, so layer L is index L, as in notebook 27
    for s in todo:
        print('features for', s, len(SETS[s])); F = extract_features(tm, tt, SETS[s], TASK, LAYERS)
        for L in LAYERS: np.save(feat_path(m_, s, L), F[L])
    for tk, s in todo_tasks:
        print('features for', s, 'task', tk); F = extract_features(tm, tt, SETS[s], tk, LAYERS)
        for L in LAYERS: np.save(feat_path(m_, s, L, tk), F[L])
    del tm; gc.collect(); torch.cuda.empty_cache()
FEAT = {m: {s: {L: np.load(feat_path(m, s, L)) for L in MODELS[m]['layers']} for s in SETS} for m in MODELS}
print('feature shapes:', {m: FEAT[m]['bipia'][MODELS[m]['layers'][0]].shape for m in MODELS})

## Stage C. Fit both probes the same way, pick each layer on validation, score every set

In [ ]:
PROBE, BEST, SC, LROWS = {}, {}, {}, []
for m in MODELS:
    probes = {L: fit_probe(FEAT[m]['train'][L], TR_Y, FEAT[m]['val'][L], VA_Y) for L in MODELS[m]['layers']}
    for L in MODELS[m]['layers']: LROWS.append(dict(model=MODELS[m]['label'], layer=L, val_AUROC=round(probes[L]['val_auroc'], 4), C=probes[L]['C']))
    BEST[m] = max(MODELS[m]['layers'], key=lambda L: probes[L]['val_auroc']); PROBE[m] = probes[BEST[m]]
    SC[m] = {s: probe_scores(PROBE[m], FEAT[m][s][BEST[m]]) for s in SETS}
    print(f"{MODELS[m]['label']}: chosen layer {BEST[m]} (val AUROC {PROBE[m]['val_auroc']:.3f})")
LAY = pd.DataFrame(LROWS); LAY.to_csv('reports/taskdrift3b_layers.csv', index=False); print(LAY.to_string(index=False))
if BEST['7b'] != 14: print(f"NOTE: the refitted 7B probe chose layer {BEST['7b']}; notebook 27 chose 14 (a tie or a near-tie on validation); the 7B rows below use layer {BEST['7b']}")

## Stage D. Static panel, the adversary with the attacker moving second, the cross-tab, and the three tasks

In [ ]:
def thr(p_ben): return guar_threshold(np.asarray(p_ben, np.float64))      # the paper's rule (Appendix A.2); returns (threshold, blocked, calibration FPR)
EVR, ADR, CTR, TKR = [], [], [], []
for m in MODELS:
    S = SC[m]; lab = MODELS[m]['label']; hosts = S['bipia'][HOST]; tg, nb_, cf = thr(hosts)
    yh = np.r_[np.ones(HIJ.sum()), np.zeros(HOST.sum())]; ph = np.r_[S['bipia'][HIJ], hosts]; yr = np.r_[np.ones(HARM.sum()), np.zeros(HOST.sum())]; pr = np.r_[S['bipia'][HARM], hosts]
    row = lambda dist, au, k, n, cal, blocked, loo=None: dict(model=lab, distribution=dist, AUROC=au, FNR_guar=fmt(k / n, *wilson(k, n)), calibration=cal, benigns_blocked=blocked, loo_FPR=loo)
    EVR.append(row('BIPIA hijack vs hosts', fmt(roc_auc_score(yh, ph), *boot_auroc(yh, ph)), int((S['bipia'][HIJ] < tg).sum()), int(HIJ.sum()), 'BIPIA hosts (778)', nb_))
    EVR.append(row('BIPIA harmful vs hosts', fmt(roc_auc_score(yr, pr), *boot_auroc(yr, pr)), int((S['bipia'][HARM] < tg).sum()), int(HARM.sum()), 'BIPIA hosts (778)', nb_))
    to, nbo, _ = thr(S['opi'][OPI_Y == 0]); mo = S['opi'][OPI_Y == 1] < to
    EVR.append(row('second source (600 sampled)', fmt(roc_auc_score(OPI_Y, S['opi']), *boot_auroc(OPI_Y, S['opi'])), int(mo.sum()), len(mo), 'its own benigns (300)', nbo))
    ta, nba, _ = thr(S['agentdojo'][AD_Y == 0]); ma = S['agentdojo'][AD_Y == 1] < ta
    EVR.append(row('AgentDojo tool outputs (129 benign, 400 sampled attacked)', fmt(roc_auc_score(AD_Y, S['agentdojo']), *boot_auroc(AD_Y, S['agentdojo'])), int(ma.sum()), len(ma), 'its own benigns (129)', nba, round(loo_fpr(S['agentdojo'][AD_Y == 0]), 4)))
    for dist, s_ in (('NotInject (over-defense)', S['notinject']), ('recipes (over-defense)', S['howto'])):
        k = int((s_ >= tg).sum()); EVR.append(dict(model=lab, distribution=dist, AUROC='n/a', FNR_guar='FPR ' + fmt(k / len(s_), *wilson(k, len(s_))), calibration='BIPIA hosts (778)', benigns_blocked=nb_, loo_FPR=None))
    # the adversary: originals, rewrites chosen against the structural detector, rewrites chosen against this probe
    A = S['adv']; sel_first = group_min_index(np.load('data/phase5v2/score_full.npy'), OWNER); sel_second = group_min_index(A, OWNER)
    for setting, idx in (('static (91 originals)', FIRST), ('attacker first (rewrites chosen against structural)', sel_first), ('attacker second (rewrites chosen against the probe)', sel_second)):
        k = int((A[idx] < tg).sum()); ADR.append(dict(model=lab, setting=setting, FNR_guar=fmt(k / len(idx), *wilson(k, len(idx)))))
    # the cross-tab: among the variants the 3B target followed (judge verdicts on disk), how many does this probe flag?
    ct = []
    for key, works in JV.items():
        src, i = key.split('|'); i = int(i); vidx = FIRST[i] if src == 'orig' else SELS[src][i]
        ct.append(dict(works=int(bool(works)), flagged=int(A[vidx] >= tg)))
    CT = pd.DataFrame(ct); f1 = CT[CT.works == 1].flagged.values.astype(bool); f0 = CT[CT.works == 0].flagged.values.astype(bool)
    p1, p0 = float(f1.mean()), float(f0.mean()); lo1, hi1 = wilson(int(f1.sum()), len(f1)); lo0, hi0 = wilson(int(f0.sum()), len(f0))
    dlo = (p1 - p0) - np.sqrt((p1 - lo1) ** 2 + (hi0 - p0) ** 2); dhi = (p1 - p0) + np.sqrt((hi1 - p1) ** 2 + (p0 - lo0) ** 2)   # Newcombe's interval for two independent proportions
    CTR.append(dict(model=lab, followed_n=len(f1), followed_flagged=fmt(p1, lo1, hi1), not_followed_n=len(f0), not_followed_flagged=fmt(p0, lo0, hi0),
                    difference=round(p1 - p0, 3), difference_lo=round(float(dlo), 3), difference_hi=round(float(dhi), 3)))
    for tk in ('summarize', 'qa', 'extract'):
        Fb = np.load(feat_path(m, 'bipia', BEST[m], tk)) if tk != 'summarize' else FEAT[m]['bipia'][BEST[m]]; s_ = probe_scores(PROBE[m], Fb); tt_, _, _ = thr(s_[HOST])
        TKR.append(dict(model=lab, task=tk, hijack_AUROC=round(roc_auc_score(yh, np.r_[s_[HIJ], s_[HOST]]), 3), harmful_AUROC=round(roc_auc_score(yr, np.r_[s_[HARM], s_[HOST]]), 3), hijack_FNR_guar=round(fnr(s_[HIJ], tt_), 3)))
EV, ADV, XT, TK = (pd.DataFrame(x) for x in (EVR, ADR, CTR, TKR))
EV.to_csv('reports/taskdrift3b_eval.csv', index=False); ADV.to_csv('reports/taskdrift3b_adversary.csv', index=False); XT.to_csv('reports/taskdrift3b_crosstab.csv', index=False); TK.to_csv('reports/taskdrift3b_tasks.csv', index=False)
for title, D_ in (('static panel (guaranteed thresholds by the paper\'s rule)', EV), ('the paraphrase adversary', ADV), ('cross-tab: share flagged among variants the 3B target followed and did not follow (Newcombe interval on the difference)', XT), ('the same probe under three tasks on BIPIA', TK)):
    print(f'\n=== {title} ==='); print(D_.to_string(index=False))
# the earlier numbers (notebook 27, 7B, earlier threshold routine) must be reproduced by the 7B rows up to the routine: the AUROCs are routine-free
OLD_EV = pd.read_csv('reports/taskdrift_eval.csv'); new7 = EV[EV.model == MODELS['7b']['label']]
for dist in ('BIPIA hijack vs hosts', 'BIPIA harmful vs hosts', 'second source (600 sampled)'):
    a = float(OLD_EV[OLD_EV.distribution == dist].AUROC.iloc[0].split()[0]); b = float(new7[new7.distribution == dist].AUROC.iloc[0].split()[0])
    assert abs(a - b) < 0.005, f'7B {dist}: notebook 27 AUROC {a}, refit {b}'
print('\nthe refitted 7B probe reproduces notebook 27\'s AUROCs')

## Stage E. Figure, log and commit

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6)); cols = {'3b': '#c44e52', '7b': '#8c8c8c'}
for m in MODELS:
    S = SC[m]; hosts = S['bipia'][HOST]
    for lab, y, p, ls in [('BIPIA hijack', np.r_[np.ones(HIJ.sum()), np.zeros(HOST.sum())], np.r_[S['bipia'][HIJ], hosts], '-'), ('AgentDojo', AD_Y, S['agentdojo'], '--'), ('second source', OPI_Y, S['opi'], ':')]:
        f, t, _ = roc_curve(y, p); axes[0].plot(f, t, ls=ls, color=cols[m], label=f"{MODELS[m]['label'].split(' (')[0]} {lab} ({roc_auc_score(y, p):.2f})")
axes[0].plot([0, 1], [0, 1], 'k--', lw=.8); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('Task-drift probe, both models'); axes[0].legend(fontsize=7); axes[0].grid(alpha=.3)
w = 0.38
for j, m in enumerate(MODELS):
    v = [float(x.split()[0]) for x in ADV[ADV.model == MODELS[m]['label']].FNR_guar]; axes[1].bar(np.arange(3) + (j - 0.5) * w, v, w, color=cols[m], label=MODELS[m]['label'].split(' (')[0])
axes[1].set_xticks(range(3)); axes[1].set_xticklabels(['static', 'attacker first', 'attacker second'], fontsize=9); axes[1].set_ylim(0, 1.05); axes[1].set_ylabel('miss rate at the guaranteed 1% host FPR'); axes[1].set_title('Under the paraphrase adversary'); axes[1].legend(fontsize=8); axes[1].grid(axis='y', alpha=.3)
for j, m in enumerate(MODELS):
    r = XT[XT.model == MODELS[m]['label']].iloc[0]; v = [float(r.not_followed_flagged.split()[0]), float(r.followed_flagged.split()[0])]
    lo = [float(r.not_followed_flagged.split('[')[1].split(',')[0]), float(r.followed_flagged.split('[')[1].split(',')[0])]; hi = [float(r.not_followed_flagged.split(', ')[1].rstrip(']')), float(r.followed_flagged.split(', ')[1].rstrip(']'))]
    axes[2].bar(np.arange(2) + (j - 0.5) * w, v, w, yerr=[[a - b for a, b in zip(v, lo)], [b - a for a, b in zip(v, hi)]], capsize=3, color=cols[m], label=MODELS[m]['label'].split(' (')[0])
axes[2].set_xticks(range(2)); axes[2].set_xticklabels(['3B did not follow', '3B followed']); axes[2].set_ylim(0, 1.05); axes[2].set_ylabel('share flagged by the probe'); axes[2].set_title('Flag rate by whether the 3B followed the injection'); axes[2].legend(fontsize=8); axes[2].grid(axis='y', alpha=.3)
plt.tight_layout(); plt.savefig('figures/taskdrift_3b.png', dpi=150, bbox_inches='tight'); plt.show()
from reslog import log_result
r3 = XT[XT.model == MODELS['3b']['label']].iloc[0]
verdict = (f"On the 3B's own states (layer {BEST['3b']}), the probe flags {r3.followed_flagged} of the variants the 3B followed and {r3.not_followed_flagged} of those it did not; "
           f"difference {r3.difference:+.3f} [{r3.difference_lo:+.3f}, {r3.difference_hi:+.3f}]. " + ('The interval excludes zero: the probe sees obedience, weakly.' if r3.difference_lo > 0 else 'The interval includes zero: the probe does not show that it sees obedience.'))
summary = '\n\n'.join([verdict, 'Layers:\n' + LAY.to_string(index=False), 'Static panel:\n' + EV.to_string(index=False), 'Adversary:\n' + ADV.to_string(index=False), 'Cross-tab:\n' + XT.to_string(index=False), 'Tasks:\n' + TK.to_string(index=False)])
log_result('nb52: the task-drift probe on Qwen2.5-3B, the model whose behaviour the cross-tab uses; the 7B refitted beside it under the same threshold rule', summary, csv_df=EV, csv_name='taskdrift3b_eval.csv')
print(summary)

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py", "nb52: task-drift probe refitted on Qwen2.5-3B-Instruct, the Section 8 target whose judge verdicts the cross-tab uses (notebook 27 read the 7B); same training and evaluation sets, same procedure, layers at the same depths; the 7B probe refitted from cached features beside it; every guaranteed threshold by the paper's final rule with the AgentDojo leave-one-out FPR; cross-tab with Wilson intervals; reports/taskdrift3b_*.csv, figures/taskdrift_3b.png"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)